# Evaluación de modelos para BugTriage AI

## Objetivo

Comparar modelos de clasificación zero-shot para determinar cuál resulta más adecuado
para clasificar reportes de bugs de software.

La evaluación se realizará utilizando exactamente los mismos casos de prueba para
todos los modelos.

### Modelos candidatos

1. `facebook/bart-large-mnli`
2. `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`

### Hipótesis

Se espera que el modelo multilingüe pueda presentar un mejor comportamiento sobre
reportes escritos en español, mientras que `facebook/bart-large-mnli` constituye
un modelo de referencia ampliamente utilizado para clasificación zero-shot.

La hipótesis será validada experimentalmente y no se asumirá que un modelo es mejor
antes de realizar las pruebas.

In [ ]:
from transformers import pipeline
import pandas as pd
import time

In [ ]:
CATEGORIAS = [
    "Funcional",
    "UI/UX",
    "Autenticación",
    "Datos",
    "Accesibilidad"
]

CATEGORIAS

['Funcional', 'UI/UX', 'Autenticación', 'Datos', 'Accesibilidad']

## Casos de prueba

Se utilizarán los mismos casos para evaluar todos los modelos candidatos.

El conjunto incluye casos normales y casos potencialmente difíciles:
- texto muy corto;
- lenguaje informal;
- lenguaje técnico/jargon;
- español e inglés;
- mezcla de idiomas;
- reportes ambiguos;
- más de un problema mencionado.

In [ ]:
casos_prueba = [
    {
        "id": 1,
        "texto": "El botón de guardar no funciona.",
        "esperado": "Funcional"
    },
    {
        "id": 2,
        "texto": "La pantalla de login se rompe cuando ingreso mi contraseña.",
        "esperado": "Autenticación"
    },
    {
        "id": 3,
        "texto": "Los nombres de los usuarios aparecen duplicados.",
        "esperado": "Datos"
    },
    {
        "id": 4,
        "texto": "El botón de continuar queda cortado en la versión móvil.",
        "esperado": "UI/UX"
    },
    {
        "id": 5,
        "texto": "El lector de pantalla no anuncia el campo de correo electrónico.",
        "esperado": "Accesibilidad"
    },
    {
        "id": 6,
        "texto": "No anda.",
        "esperado": "Funcional"
    },
    {
        "id": 7,
        "texto": "Después del último deploy rompieron el flow de login.",
        "esperado": "Autenticación"
    },
    {
        "id": 8,
        "texto": "The submit button is broken on mobile.",
        "esperado": "UI/UX"
    },
    {
        "id": 9,
        "texto": "El login tira error después del deploy.",
        "esperado": "Autenticación"
    },
    {
        "id": 10,
        "texto": "Desde el celular no puedo terminar el registro.",
        "esperado": "UI/UX"
    },
    {
        "id": 11,
        "texto": "Che, cada vez que guardo se me borra todo.",
        "esperado": "Datos"
    },
    {
        "id": 12,
        "texto": "El login anda perfecto, pero el lector no me lee el botón.",
        "esperado": "Accesibilidad"
    }
]

df_casos = pd.DataFrame(casos_prueba)

df_casos


,id,texto,esperado
0,1,El botón de guardar no funciona.,Funcional
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación
2,3,Los nombres de los usuarios aparecen duplicados.,Datos
3,4,El botón de continuar queda cortado en la vers...,UI/UX
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad
5,6,No anda.,Funcional
6,7,Después del último deploy rompieron el flow de...,Autenticación
7,8,The submit button is broken on mobile.,UI/UX
8,9,El login tira error después del deploy.,Autenticación
9,10,Desde el celular no puedo terminar el registro.,UI/UX


In [ ]:
casos_dificiles = {
    6: "Muy corto",
    7: "Jargon técnico",
    8: "Otro idioma",
    9: "Mezcla de lenguaje técnico",
    10: "Potencialmente ambiguo",
    11: "Lenguaje informal",
    12: "Dos conceptos en el mismo reporte"
}

casos_dificiles

{6: 'Muy corto',
 7: 'Jargon técnico',
 8: 'Otro idioma',
 9: 'Mezcla de lenguaje técnico',
 10: 'Potencialmente ambiguo',
 11: 'Lenguaje informal',
 12: 'Dos conceptos en el mismo reporte'}

### Casos difíciles

Los casos 6 a 12 permiten observar limitaciones del modelo frente a situaciones
que pueden ser más difíciles que una clasificación directa.

En particular, se incluyen textos muy cortos, terminología utilizada en equipos
de desarrollo, lenguaje informal, inglés, mezcla de idiomas y reportes que
contienen más de un elemento potencialmente relevante.



## PROBAR MODELOS

## Modelo 1 — facebook/bart-large-mnli

Este modelo utiliza Natural Language Inference (NLI) para realizar clasificación
zero-shot.

Se evaluará utilizando las mismas categorías y los mismos casos definidos
anteriormente.

In [34]:
modelo_1 = "facebook/bart-large-mnli"

inicio = time.perf_counter()

clasificador_1 = pipeline(
    "zero-shot-classification",
    model=modelo_1
)

tiempo_carga_1 = time.perf_counter() - inicio

print("Modelo cargado:", modelo_1)
print(f"Tiempo de carga: {tiempo_carga_1:.2f} segundos")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Modelo cargado: facebook/bart-large-mnli
Tiempo de carga: 5.86 segundos


In [35]:
resultado_prueba_1 = clasificador_1(
    casos_prueba[0]["texto"],
    CATEGORIAS
)

resultado_prueba_1

{'sequence': 'El botón de guardar no funciona.',
 'labels': ['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional'],
 'scores': [0.26611194014549255,
  0.26195088028907776,
  0.2077663242816925,
  0.19364923238754272,
  0.07052168250083923]}

In [ ]:
type(resultado_prueba_1)


dict

In [ ]:
resultado_prueba_1["labels"]

['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional']

In [ ]:
resultado_prueba_1["scores"]

[0.26611194014549255,
 0.26195088028907776,
 0.2077663242816925,
 0.19364923238754272,
 0.07052168250083923]

In [13]:
resultados_modelo_1 = []

inicio_total_1 = time.perf_counter()

for caso in casos_prueba:
    inicio = time.perf_counter()

    resultado = clasificador_1(
        caso["texto"],
        CATEGORIAS
    )

    tiempo = time.perf_counter() - inicio

    prediccion = resultado["labels"][0]
    score = resultado["scores"][0]

    resultados_modelo_1.append({
        "id": caso["id"],
        "frase": caso["texto"],
        "esperado": caso["esperado"],
        "prediccion": prediccion,
        "score": score,
        "correcto": prediccion == caso["esperado"],
        "tiempo_segundos": tiempo
    })

tiempo_total_1 = time.perf_counter() - inicio_total_1

df_modelo_1 = pd.DataFrame(resultados_modelo_1)

df_modelo_1

,id,frase,esperado,prediccion,score,correcto,tiempo_segundos
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112,False,1.001640
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Autenticación,0.307510,True,0.896505
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,Datos,0.325560,True,0.879123
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505,False,0.929636
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,Accesibilidad,0.281191,True,0.957054
5,6,No anda.,Funcional,Datos,0.409652,False,0.838082
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505,False,0.869987
7,8,The submit button is broken on mobile.,UI/UX,UI/UX,0.622319,True,0.818239
8,9,El login tira error después del deploy.,Autenticación,Autenticación,0.296833,True,0.866580
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046,False,0.861106


In [14]:
precision_1 = df_modelo_1["correcto"].mean()

print(f"Precisión: {precision_1:.2%}")
print(f"Tiempo total: {tiempo_total_1:.2f} segundos")
print(f"Tiempo promedio: {df_modelo_1['tiempo_segundos'].mean():.2f} segundos")

Precisión: 50.00%
Tiempo total: 10.79 segundos
Tiempo promedio: 0.90 segundos


In [16]:
df_modelo_1.to_csv(
    "resultados_bart_large_mnli.csv",
    index=False
)

In [28]:
df_modelo_1[
    ["id", "frase", "esperado", "prediccion", "score", "correcto"]
]

,id,frase,esperado,prediccion,score,correcto
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112,False
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Autenticación,0.307510,True
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,Datos,0.325560,True
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505,False
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,Accesibilidad,0.281191,True
5,6,No anda.,Funcional,Datos,0.409652,False
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505,False
7,8,The submit button is broken on mobile.,UI/UX,UI/UX,0.622319,True
8,9,El login tira error después del deploy.,Autenticación,Autenticación,0.296833,True
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046,False


In [29]:
print("ERRORES DEL MODELO 1")

df_modelo_1[
    df_modelo_1["correcto"] == False
][
    ["id", "frase", "esperado", "prediccion", "score"]
]

ERRORES DEL MODELO 1


,id,frase,esperado,prediccion,score
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505
5,6,No anda.,Funcional,Datos,0.409652
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046
10,11,"Che, cada vez que guardo se me borra todo.",Datos,Autenticación,0.374223


## Modelo 2 — MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7

Este modelo fue seleccionado como candidato multilingüe para evaluar su comportamiento
sobre reportes de bugs escritos principalmente en español.

Al igual que el modelo anterior, se utilizará para clasificación zero-shot mediante
la tarea `zero-shot-classification`.

Se mantendrán exactamente las mismas categorías y los mismos 12 casos de prueba
para que la comparación sea equivalente.

In [17]:
modelo_2 = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

inicio = time.perf_counter()

clasificador_2 = pipeline(
    "zero-shot-classification",
    model=modelo_2
)

tiempo_carga_2 = time.perf_counter() - inicio

print("Modelo cargado:", modelo_2)
print(f"Tiempo de carga: {tiempo_carga_2:.2f} segundos")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

D:\TECNICATURA IA\seminario de actualizacion 2026\proyecto_integrador\TP_Integrador\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Modelo cargado: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Tiempo de carga: 41.32 segundos


In [18]:
resultado_prueba_2 = clasificador_2(
    casos_prueba[0]["texto"],
    CATEGORIAS
)

resultado_prueba_2

{'sequence': 'El botón de guardar no funciona.',
 'labels': ['UI/UX', 'Accesibilidad', 'Datos', 'Autenticación', 'Funcional'],
 'scores': [0.45136383175849915,
  0.2290823608636856,
  0.18632791936397552,
  0.09525997191667557,
  0.03796588256955147]}

In [19]:
type(resultado_prueba_2)


dict

In [20]:
resultado_prueba_2["labels"]

['UI/UX', 'Accesibilidad', 'Datos', 'Autenticación', 'Funcional']

In [21]:
resultado_prueba_2["scores"]

[0.45136383175849915,
 0.2290823608636856,
 0.18632791936397552,
 0.09525997191667557,
 0.03796588256955147]

In [25]:
resultados_modelo_2 = []

inicio_total_2 = time.perf_counter()

for caso in casos_prueba:
    inicio = time.perf_counter()

    resultado = clasificador_2(
        caso["texto"],
        CATEGORIAS
    )

    tiempo = time.perf_counter() - inicio

    prediccion = resultado["labels"][0]
    score = resultado["scores"][0]

    resultados_modelo_2.append({
        "id": caso["id"],
        "frase": caso["texto"],
        "esperado": caso["esperado"],
        "prediccion": prediccion,
        "score": score,
        "correcto": prediccion == caso["esperado"],
        "tiempo_segundos": tiempo
    })

tiempo_total_2 = time.perf_counter() - inicio_total_2

df_modelo_2 = pd.DataFrame(resultados_modelo_2)

df_modelo_2

,id,frase,esperado,prediccion,score,correcto,tiempo_segundos
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.451364,False,4.130934
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Accesibilidad,0.709738,False,4.404814
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,UI/UX,0.364299,False,3.948743
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Funcional,0.311254,False,4.310154
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,UI/UX,0.600210,False,4.473136
5,6,No anda.,Funcional,Accesibilidad,0.281979,False,3.688059
6,7,Después del último deploy rompieron el flow de...,Autenticación,Accesibilidad,0.614365,False,4.089429
7,8,The submit button is broken on mobile.,UI/UX,Accesibilidad,0.537123,False,4.181588
8,9,El login tira error después del deploy.,Autenticación,Accesibilidad,0.694260,False,3.952380
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.333061,False,4.395223


In [26]:
precision_2 = df_modelo_2["correcto"].mean()

print(f"Precisión: {precision_2:.2%}")
print(f"Tiempo total: {tiempo_total_2:.2f} segundos")
print(f"Tiempo promedio: {df_modelo_2['tiempo_segundos'].mean():.2f} segundos")

Precisión: 16.67%
Tiempo total: 49.82 segundos
Tiempo promedio: 4.15 segundos


In [27]:
df_modelo_2.to_csv(
    "resultados_mdeberta_multilingual.csv",
    index=False
)

In [30]:
print("ERRORES DEL MODELO 2")

df_modelo_2[
    df_modelo_2["correcto"] == False
][
    ["id", "frase", "esperado", "prediccion", "score"]
]

ERRORES DEL MODELO 2


,id,frase,esperado,prediccion,score
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.451364
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Accesibilidad,0.709738
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,UI/UX,0.364299
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Funcional,0.311254
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,UI/UX,0.600210
5,6,No anda.,Funcional,Accesibilidad,0.281979
6,7,Después del último deploy rompieron el flow de...,Autenticación,Accesibilidad,0.614365
7,8,The submit button is broken on mobile.,UI/UX,Accesibilidad,0.537123
8,9,El login tira error después del deploy.,Autenticación,Accesibilidad,0.694260
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.333061


In [31]:
comparacion = pd.DataFrame({
    "Modelo": [
        modelo_1,
        modelo_2
    ],
    "Precisión": [
        precision_1,
        precision_2
    ],
    "Tiempo de carga (s)": [
        tiempo_carga_1,
        tiempo_carga_2
    ],
    "Tiempo total (s)": [
        tiempo_total_1,
        tiempo_total_2
    ],
    "Tiempo promedio por caso (s)": [
        df_modelo_1["tiempo_segundos"].mean(),
        df_modelo_2["tiempo_segundos"].mean()
    ]
})

comparacion

,Modelo,Precisión,Tiempo de carga (s),Tiempo total (s),Tiempo promedio por caso (s)
0,facebook/bart-large-mnli,0.500000,55.974195,10.790579,0.899189
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,0.166667,41.322570,49.819124,4.151561


In [32]:
comparacion.sort_values(
    by="Precisión",
    ascending=False
)

,Modelo,Precisión,Tiempo de carga (s),Tiempo total (s),Tiempo promedio por caso (s)
0,facebook/bart-large-mnli,0.500000,55.974195,10.790579,0.899189
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,0.166667,41.322570,49.819124,4.151561


## Análisis preliminar de resultados

El benchmark mostró una diferencia significativa entre los dos modelos evaluados.

`facebook/bart-large-mnli` obtuvo una precisión del 50,00% (6 de 12 casos),
mientras que `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`
obtuvo una precisión del 16,67% (2 de 12 casos).

Además, BART presentó un tiempo promedio de inferencia de 0,90 segundos
por caso, frente a 4,15 segundos del modelo multilingüe.

Por lo tanto, para este conjunto de prueba y bajo las condiciones del
experimento, BART presenta el mejor equilibrio entre precisión y tiempo
de inferencia.

Sin embargo, los resultados no permiten afirmar que BART sea superior
en términos generales. El conjunto utilizado contiene solamente 12 casos
y funciona como benchmark exploratorio para seleccionar un modelo candidato.

Los errores observados se concentran especialmente en reportes muy cortos,
ambiguos, informales o con terminología técnica. Esto muestra una limitación
importante de la clasificación zero-shot: la predicción depende de la
relación semántica entre el texto y las etiquetas proporcionadas, y no de
un modelo entrenado específicamente para las categorías de BugTriage AI.

In [36]:
import os

token = os.getenv("HF_TOKEN")

if token:
    print("Token disponible para el notebook.")
else:
    print("ERROR: HF_TOKEN no está disponible.")

Token disponible para el notebook.


In [8]:
import os

token = os.getenv("HF_TOKEN")

print("Token cargado:", token is not None)

Token cargado: True


In [37]:
from huggingface_hub import InferenceClient

In [38]:
modelo_1 = "facebook/bart-large-mnli"

In [39]:
client = InferenceClient(
    model=modelo_1,
    token=token
)

In [40]:

texto_prueba = casos_prueba[0]["texto"]

texto_prueba

'El botón de guardar no funciona.'

In [41]:
resultado_remoto = client.zero_shot_classification(
    texto_prueba,
    candidate_labels=CATEGORIAS,
    multi_label=False
)

resultado_remoto

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/hf-inference/models/facebook/bart-large-mnli' (Request ID: Root=1-6ac6e151-41ffce7b211f70851c53559b;065bf6da-c698-455e-ac68-c8efeac83387)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

## Error encontrado: InferenceClient y créditos de Hugging Face

Al intentar ejecutar el mismo modelo mediante `InferenceClient`, utilizando
la tarea `zero-shot-classification`, la solicitud llegó correctamente al
servicio de Hugging Face pero fue rechazada con el código HTTP 402
`Payment Required`.

El mensaje recibido indicó que no quedaban créditos disponibles para
Inference Providers.

### Causa

El problema no estaba relacionado con el código de clasificación ni con
el modelo. La llamada utilizaba correctamente el método
`zero_shot_classification()` y el modelo `facebook/bart-large-mnli`.

La limitación correspondía a la disponibilidad de créditos de la cuenta
para realizar inferencia remota mediante Inference Providers.

### Consecuencia

No fue posible completar el benchmark remoto de los 12 casos sin utilizar
créditos adicionales.

Por este motivo, el experimento local mediante `pipeline()` se utiliza
como benchmark cuantitativo principal, mientras que la prueba con
`InferenceClient` queda documentada como una prueba de integración que
permitió verificar una limitación real del servicio.

Este comportamiento también se considera al analizar la arquitectura final
de la aplicación.

## Comparación de las formas de inferencia

### Pipeline

Con `pipeline()` el modelo se descargó y ejecutó localmente.

Ventajas observadas:
- no depende de una llamada remota para cada predicción;
- permite ejecutar el modelo sin consumir créditos de Inference Providers;
- permite medir directamente el tiempo de inferencia local.

Desventaja:
- el entorno donde se ejecuta la aplicación debe disponer de los recursos
  necesarios para cargar los pesos del modelo.

### InferenceClient

Con `InferenceClient` la aplicación envía la solicitud a la infraestructura
de inferencia de Hugging Face.

Ventaja:
- el servidor de la aplicación no necesita descargar y ejecutar localmente
  los pesos del modelo.

Limitación observada:
- la disponibilidad de la inferencia remota depende del servicio y de los
  créditos disponibles para Inference Providers.

In [42]:
def clasificar_local(texto, categorias):
    return clasificador_1(
        texto,
        candidate_labels=categorias,
        multi_label=False
    )

In [44]:
resultado_local = clasificar_local(
    texto_prueba,
    CATEGORIAS
)

resultado_local

{'sequence': 'El botón de guardar no funciona.',
 'labels': ['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional'],
 'scores': [0.26611194014549255,
  0.26195088028907776,
  0.2077663242816925,
  0.19364923238754272,
  0.07052168250083923]}

## Comparación entre inferencia local y remota

En este proyecto se probaron dos formas de ejecutar el modelo de clasificación.

### Inferencia local con `pipeline()`

Mediante `pipeline()` de Hugging Face Transformers, el modelo `facebook/bart-large-mnli` se descarga y sus pesos se ejecutan localmente en el equipo.

La función `clasificar_local()` devuelve un diccionario con la secuencia analizada, las categorías ordenadas según su puntuación y los scores correspondientes.

En este caso, para la frase:

> "El botón de guardar no funciona."

el modelo predijo `UI/UX` con un score de aproximadamente 0,2661.

Esta predicción coincide con el benchmark realizado anteriormente y fue incorrecta respecto de la categoría esperada, `Funcional`.

### Inferencia remota con `InferenceClient`

También se intentó ejecutar el mismo modelo mediante `InferenceClient`, utilizando el servicio de inferencia de Hugging Face.

La primera prueba utilizó incorrectamente `text_classification()`, pero el modelo requiere la tarea `zero-shot-classification`. Luego se corrigió la llamada utilizando `zero_shot_classification()`.

La llamada corregida llegó al servicio de Hugging Face, pero devolvió un error HTTP 402 (`Payment Required`) debido a que la cuenta no disponía de créditos restantes para Inference Providers.

Por este motivo, el benchmark cuantitativo principal se realizó mediante inferencia local con `pipeline()`, mientras que `InferenceClient` se documenta como una prueba de integración remota y como evidencia de una limitación real del servicio.

## Ficha técnica del modelo 1: facebook/bart-large-mnli

### Tarea

`facebook/bart-large-mnli` es un modelo basado en BART ajustado para Natural Language Inference (NLI) sobre el dataset MultiNLI.

Hugging Face lo presenta como un modelo apto para clasificación zero-shot. En este enfoque, las categorías propuestas por la aplicación se convierten en hipótesis y el modelo evalúa la relación entre el reporte y cada categoría.

### Funcionamiento en BugTriage AI

En lugar de entrenar el modelo específicamente con reportes de bugs, se le proporciona un conjunto de categorías:

- Funcional
- UI/UX
- Autenticación
- Datos
- Accesibilidad

El modelo calcula qué categoría presenta mayor relación semántica con el reporte recibido.

### Idioma

El modelo BART base fue preentrenado sobre inglés y `bart-large-mnli` fue ajustado sobre MultiNLI. Por este motivo, su comportamiento puede ser menos adecuado para textos en español que el de un modelo específicamente multilingüe.

Esto es relevante para BugTriage AI porque los reportes utilizados en este proyecto están principalmente escritos en español.

### Datos de entrenamiento

El checkpoint `bart-large-mnli` fue ajustado utilizando el dataset MultiNLI (Multi-Genre Natural Language Inference).

Por lo tanto, el modelo fue entrenado para resolver relaciones de inferencia textual y no para clasificar específicamente tickets de QA en las categorías utilizadas por BugTriage AI.

### Tamaño

BART-large posee aproximadamente 406 millones de parámetros. El archivo principal de pesos en formato `safetensors` ocupa aproximadamente 1,63 GB.

Esto tiene impacto directo en el despliegue: ejecutar el modelo localmente requiere más memoria y almacenamiento que utilizar un modelo pequeño o realizar la inferencia mediante un servicio remoto.

### Licencia

Licencia MIT.

### Limitaciones relevantes para el proyecto

El modelo no fue entrenado específicamente con reportes de errores de software ni con las cinco categorías de BugTriage AI.

Por esta razón, puede confundir conceptos relacionados semánticamente, especialmente cuando el reporte es muy corto, ambiguo, informal o utiliza terminología específica del desarrollo de software.

Además, el modelo no debe interpretarse como un clasificador especializado de QA. La clasificación obtenida es una inferencia zero-shot y debe considerarse como asistencia para una primera etapa de triage, no como una decisión definitiva.

## Ficha técnica del modelo 2: mDeBERTa-v3-base-xnli-multilingual-nli-2mil7

### Tarea

`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` es un modelo multilingüe orientado a Natural Language Inference (NLI) y clasificación zero-shot.

### Funcionamiento en BugTriage AI

Al igual que BART, el modelo recibe el reporte y las categorías candidatas.

Para cada categoría evalúa la relación semántica entre el texto del reporte y la hipótesis asociada a esa categoría. Finalmente devuelve las categorías ordenadas según su puntuación.

### Idiomas

El modelo puede realizar NLI sobre 100 idiomas.

Sin embargo, su entrenamiento específico para NLI se realizó sobre 27 idiomas. El español (`es`) forma parte de esos idiomas.

Esto representa una ventaja teórica para BugTriage AI, ya que la mayoría de los reportes utilizados por la aplicación están escritos en español.

### Datos de entrenamiento

El modelo base mDeBERTa-v3 fue preentrenado sobre CC100, un conjunto de datos multilingüe.

Posteriormente, el modelo fue ajustado utilizando XNLI y el dataset `multilingual-nli-26lang-2mil7`.

El dataset multilingüe utilizado para el ajuste contiene aproximadamente 2.730.000 pares de hipótesis y premisas en 26 idiomas, además de una muestra en inglés.

El conjunto total utilizado durante el entrenamiento del modelo alcanzó aproximadamente 3.287.280 pares.

### Tamaño

El Model Card informa aproximadamente 0,3 billones de parámetros (0,3B).

Es un modelo de tamaño base y, por cantidad de parámetros, es menor que BART-large.

### Licencia

Licencia MIT.

### Limitaciones relevantes para el proyecto

Aunque el modelo tiene una cobertura multilingüe superior y fue ajustado específicamente para NLI en español, esto no garantiza una mejor clasificación en el dominio de los reportes de software.

El propio Model Card advierte que parte del dataset multilingüe fue generado mediante traducción automática. Esto puede introducir problemas de calidad en datos utilizados para una tarea compleja como NLI.

Además, el modelo tampoco fue entrenado específicamente para clasificar bugs en las cinco categorías de BugTriage AI.

### Resultado en nuestro benchmark

A pesar de sus ventajas multilingües, el modelo obtuvo una precisión inferior en el conjunto de prueba utilizado en este proyecto:

- Precisión: 16,67 %
- Casos correctos: 2 de 12
- Tiempo promedio de inferencia: 4,15 segundos por caso

Por lo tanto, en este experimento no fue seleccionado para la aplicación final.

## Comparación y selección del modelo

La selección del modelo se realizó mediante un benchmark controlado utilizando exactamente los mismos 12 reportes de prueba y las mismas cinco categorías.

| Modelo | Precisión | Tiempo promedio | Resultado |
|---|---:|---:|---|
| `facebook/bart-large-mnli` | 50,00 % | 0,90 s | Seleccionado |
| `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` | 16,67 % | 4,15 s | Descartado |

El modelo `facebook/bart-large-mnli` obtuvo 6 predicciones correctas de 12, mientras que el modelo multilingüe obtuvo 2 de 12.

Además, BART presentó un tiempo promedio de inferencia aproximadamente 4,6 veces menor en las condiciones de este experimento.

Por lo tanto, se selecciona `facebook/bart-large-mnli` para la versión final de BugTriage AI.

Esta decisión no significa que BART sea universalmente superior a mDeBERTa. Significa que presentó el mejor comportamiento entre los modelos evaluados utilizando el conjunto de prueba definido para este proyecto.

La cantidad de casos utilizada es pequeña, por lo que el resultado debe considerarse un benchmark exploratorio y no una evaluación estadística definitiva.
